## Measuring Cache Effectiveness

### Setup Environment

In [ ]:
import sys
!{sys.executable} -m pip install matplotlib panel jaro-winkler seaborn fuzzywuzzy

In [14]:
import pandas as pd
import numpy as np
import time

from tqdm.auto import tqdm

from cache.evals import CacheEvaluator
from cache.faq_data_container import FAQDataContainer
from cache.wrapper import SemanticCacheWrapper
from cache.config import config, load_openai_key

In [15]:
data_container = FAQDataContainer()
faq_df, test_df = data_container.faq_df, data_container.test_df

test_queries = test_df["question"].tolist()

Loaded 8 FAQ entries
Loaded 80 test queries


In [17]:
import torch
# torch.compile (Dynamo) doesn't support Python 3.12 in torch<2.3.
# Patch it to a no-op so ModernBertModel can be imported; inference still works.
torch.compile = lambda fn=None, **kwargs: (fn if fn is not None else lambda f: f)


In [18]:
cache_wrapper = SemanticCacheWrapper.from_config(config)

✅ Redis is running and accessible!


ResponseError: unknown command 'FT._LIST', with args beginning with: 

## Evaluating cache quality

In [19]:
# Cache hydration via wrapper helper
cache_wrapper.hydrate_from_df(faq_df)

cache_wrapper.check(faq_df["question"].iloc[0])

cache_results = cache_wrapper.check_many(test_queries)
cache_results[:4]

NameError: name 'cache_wrapper' is not defined

In [ ]:
evaluator = CacheEvaluator(
    true_labels=data_container.label_cache_hits(cache_results),
    cache_results=cache_results,
)
evaluator.report_metrics()

In [ ]:
[[tn, fp], [fn, tp]] = evaluator.get_metrics()["confusion_mask"]
tn[:9] # true negatives

In [ ]:
evaluator.matches_df()[fp] #False Postives

### Evaluating cache latency

In [21]:
def simulate_llm_call(prompt):
    time.sleep(np.random.uniform(0.2, 0.5))
    return f"LLM response to {prompt}"

In [ ]:
from cache.evals import PerfEval

perf_eval = PerfEval()

with perf_eval:
    for query in tqdm(test_queries):
        cache_wrapper.check(query)
        perf_eval.tick("cache_hit")
        perf_eval.start()
        simulate_llm_call(query)
        perf_eval.tick("llm_call")

metrics = perf_eval.get_metrics(labels=["cache_hit", "llm_call"])

In [ ]:
metrics["by_label"]

In [ ]:
perf_eval.plot(
    title="Performance Comparison", show_cost_analysis=False
)

In [ ]:
llm_latency = metrics["by_label"]["llm_call"]["average_latency"]
cache_latency = metrics["by_label"]["cache_hit"]["average_latency"]

cache_hit_rate = 0.3
cached_llm_latency = llm_latency * (1 - cache_hit_rate) + cache_latency * cache_hit_rate
cached_llm_drop_in_latency = (llm_latency - cached_llm_latency) / llm_latency
cached_llm_speedup = llm_latency / cached_llm_latency
print(f"Overall latency drop of an LLM app: {int(cached_llm_drop_in_latency * 100)}%")
print(f"Overall speedup of an LLM app {cached_llm_speedup:.2f}x")

### LLM-as-a-Judge for cache quality evaluation

In [ ]:
cache_wrapper.hydrate_from_df(faq_df)

# we set the distance to obtain even bad matches and evaluate if they are true negatives
full_retrieval_nearest_neighbors = cache_wrapper.check_many(
    test_queries, distance_threshold=1
)
full_retrieval_matches = [h.matches[0].prompt for h in full_retrieval_nearest_neighbors]
full_retrieval_matches[:4]

In [ ]:
load_openai_key()

In [ ]:
from cache.llm_evaluator import LLMEvaluator

evaluator = LLMEvaluator.construct_with_gpt()

In [ ]:
llm_similarity_results = evaluator.predict(
    dataset=zip(test_queries, full_retrieval_matches),
    batch_size=5,
)
llm_similarity_results.df.head()

In [ ]:
# When evaluation is based on full retrieval we should use this constructor
evaluator = CacheEvaluator.from_full_retrieval(
    true_labels=llm_similarity_results.df["is_similar"].values,
    cache_results=cache_wrapper.check_many(test_queries),
)
evaluator.report_metrics()

In [ ]:
cache_wrapper.cache.clear()